<a href="https://colab.research.google.com/github/iurkrvvr-collab/lab1/blob/main/text_stats_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install tokenize_uk pymorphy3 pymorphy3-dicts-uk

In [22]:
"""
Статистика кількох текстів: SQLite + агрегуючі запити (SELECT ... COUNT(...) ... GROUP BY).

  1. Токенізація (tokenize_uk), лематизація і частини мови (pymorphy) -> texts.db
     Кожне слововживання - окремий рядок (повтори НЕ ігноруються).
  2. Кожен текст обрізається до кратного 1000 слововживань (максимум 20 000) і ділиться на
     РІВНІ підвибірки по 1000 слів. Неповний хвіст відкидається.
  3. Частотні словники (GROUP BY): freq_forms, freq_lemmas, freq_pos
     (id, абс. частота, частота в кожній підвибірці part_1..part_N).
  4. Статистики: частини мови, частки, відносна похибка (ЧМ, леми, словоформи), TF-IDF
     (документ = текст; і документ = підвибірка), TTR, індекс Гіро, довжина слів/речень, hapax.
  5. CSV у output/, чернетка звіту output/report.md.

Colab (окрема комірка):
    !pip install tokenize_uk pymorphy3 pymorphy3-dicts-uk beautifulsoup4 requests

Власні тексти: texts/<назва>.txt. Щоб склеїти кілька файлів в одну вибірку:
    texts/kotsiubynskyi__fata_morgana.txt, texts/kotsiubynskyi__tini.txt ...
    (все до подвійного підкреслення - назва вибірки).
"""
import csv
import glob
import math
import os
import re
import sqlite3
import sys

import requests
from bs4 import BeautifulSoup
import tokenize_uk as tok

try:
    import pymorphy2 as pm
except Exception:
    import pymorphy3 as pm

MORPH = pm.MorphAnalyzer(lang="uk")

SUBSAMPLE_SIZE = 1000
MAX_SUBSAMPLES = 20
TARGET = SUBSAMPLE_SIZE * MAX_SUBSAMPLES
OUT_DIR = "output"


SCHEMA = """
CREATE TABLE IF NOT EXISTS texts (
    text_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title   TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS sentences (
    sentence_id INTEGER PRIMARY KEY AUTOINCREMENT,
    text_id     INTEGER NOT NULL REFERENCES texts(text_id),
    position    INTEGER NOT NULL,
    content     TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS words (
    word_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    sentence_id INTEGER NOT NULL REFERENCES sentences(sentence_id),
    text_id     INTEGER NOT NULL REFERENCES texts(text_id),
    position    INTEGER NOT NULL,   -- позиція в реченні
    word        TEXT NOT NULL,      -- як у тексті
    clean_word  TEXT NOT NULL,      -- нижній регістр
    lemma       TEXT NOT NULL,
    pos         TEXT NOT NULL,      -- NOUN, VERB, ADJF, ... (UNKN - не розпізнано)
    subsample   INTEGER NOT NULL    -- номер підвибірки (1..N) у межах тексту
);
CREATE INDEX IF NOT EXISTS idx_words_text  ON words(text_id);
CREATE INDEX IF NOT EXISTS idx_words_lemma ON words(lemma);
"""

SAMPLE_TEXTS = {
    "Погода": ("Того року літня погода затримала свій відхід надовго, а осінь заступила "
               "несміливо. Сонце гріло ще довго, і ліс довго стояв зелений."),
    "Наука": ("Машинне навчання - це цікава область. Навчання з учителем - один із "
              "ключових аспектів машинного навчання."),
}


HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}
MIN_WORDS = 1500

META = {
    "kotsiubynskyi": ("Проза М. Коцюбинського («Intermezzo» та ін.)",
                      "відпочинок утомленого письменника серед природи, роздуми про людину і світ",
                      "імпресіоністична новела (1908)", "художній",
                      "https://osvita.ua/school/literature/k/63982/ (повний текст, 3 сторінки)"),
    "konstytutsiia": ("Конституція України (основний текст)",
                      "державний устрій, права, свободи й обов'язки людини і громадянина",
                      "нормативно-правовий акт (1996)", "офіційно-діловий",
                      "https://osvita.ua/legislation/law/2227/ (офіційний текст: zakon.rada.gov.ua/laws/show/254к/96-вр)"),
    "wikipedia_ml":  ("Статті «Машинне навчання», «Штучний інтелект» та ін. (укр. Вікіпедія)",
                      "машинне навчання та штучний інтелект",
                      "енциклопедична стаття", "науково-популярний",
                      "https://uk.wikipedia.org (ліцензія CC BY-SA 4.0)"),
}

POS_UK = {"NOUN": "іменник", "VERB": "дієслово", "INFN": "інфінітив", "ADJF": "прикметник",
          "ADJS": "прикметник (коротка ф.)", "COMP": "компаратив", "PRTF": "дієприкметник",
          "PRTS": "дієприкметник (коротка ф.)", "GRND": "дієприслівник", "NUMR": "числівник",
          "ADVB": "прислівник", "NPRO": "займенник", "PRED": "предикатив", "PREP": "прийменник",
          "CONJ": "сполучник", "PRCL": "частка", "INTJ": "вигук", "UNKN": "не визначено"}


def _get(url, **params):
    r = requests.get(url, headers=HEADERS, params=params or None, timeout=40)
    r.raise_for_status()
    r.encoding = "utf-8"
    return r


def _check(name, text):
    n = len(text.split())
    if n < MIN_WORDS:
        raise ValueError(f"{name}: отримано лише {n} слів - схоже, сайт віддав не те")
    return text


def fetch_kotsiubynskyi():
    base = "https://osvita.ua/school/literature/k/63982/"
    chunks = []
    for url in (base, base + "list-1.html", base + "list-2.html"):
        soup = BeautifulSoup(_get(url).text, "html.parser")
        for tag in soup(["script", "style", "nav", "header", "footer"]):
            tag.decompose()
        lines = [l.strip() for l in soup.get_text("\n").split("\n") if l.strip()]
        start = next(i for i, l in enumerate(lines) if l.startswith("Читати онлайн новелу")) + 1
        end = next((i for i in range(start, len(lines))
                    if lines[i].startswith(("(Продовження", "Скачати", "Інші твори"))), len(lines))
        chunks += [l for l in lines[start:end] if not re.fullmatch(r"A|\d+\s*c\.?", l)]
    return _check("Intermezzo", "\n".join(chunks))


def _constitution_body(text):
    """Основний текст Конституції (розділи I-XIV) без службових приміток і заголовків."""
    text = re.sub(r"\{[^{}]*\}", " ", text, flags=re.S)
    text = re.sub(r"\([^()]*(?:Закону N|Закону України|Рішенн|в редакції)[^()]*\)", " ", text)
    starts = [m.start() for m in re.finditer(r"Розділ\s+I\s+ЗАГАЛЬНІ\s+ЗАСАДИ", text)]
    s0 = starts[-1]
    m_end = re.search(r"Розділ\s+XV\s+ПЕРЕХІДНІ", text[s0:])
    e0 = s0 + m_end.start() if m_end else len(text)
    lines = []
    for l in (x.strip() for x in text[s0:e0].split("\n")):
        if not l or l.isupper() or re.match(r"Розділ\s+[IVX]+", l):
            continue
        l = re.sub(r"^Стаття\s+\d+[-\d]*\.?\s*", "", l)
        if l:
            lines.append(l)
    return "\n".join(lines)


def fetch_konstytutsiia():
    sources = ("https://osvita.ua/legislation/law/2227/",
               "https://zakon.rada.gov.ua/laws/show/254%D0%BA/96-%D0%B2%D1%80/print")
    last_err = None
    for url in sources:
        try:
            soup = BeautifulSoup(_get(url).text, "html.parser")
            for tag in soup(["script", "style"]):
                tag.decompose()
            return _check("Конституція", _constitution_body(soup.get_text("\n")))
        except Exception as err:
            print(f"   ! {url.split('/')[2]}: {err}")
            last_err = err
    raise last_err


def fetch_wikipedia(target_words=TARGET + 6000):
    titles = ["Машинне навчання", "Штучний інтелект", "Глибоке навчання",
              "Штучна нейронна мережа", "Обробка природної мови", "Навчання з учителем",
              "Навчання без учителя", "Навчання з підкріпленням", "Кластерний аналіз",
              "Класифікація (машинне навчання)", "Дерево рішень", "Метод опорних векторів",
              "Згорткова нейронна мережа", "Рекурентна нейронна мережа", "Комп'ютерний зір",
              "Велика мовна модель", "Інтелектуальний аналіз даних", "Великі дані",
              "Регресійний аналіз", "Градієнтний спуск", "Перенавчання", "Наука про дані"]
    parts, total = [], 0
    for title in titles:
        try:
            data = _get("https://uk.wikipedia.org/w/api.php", action="query", prop="extracts",
                        explaintext=1, redirects=1, titles=title, format="json").json()
            page = next(iter(data["query"]["pages"].values()))
            text = page.get("extract", "")
            if not text:
                print(f"   ! статті «{title}» немає, пропускаю")
                continue
        except Exception as err:
            print(f"   ! стаття «{title}» пропущена: {err}")
            continue
        text = re.split(r"\n==\s*(Примітки|Посилання|Література|Джерела|Див\. також|"
                        r"Зовнішні посилання)\s*==", text)[0]
        text = "\n".join(l for l in text.split("\n") if l.strip() and not re.fullmatch(r"=+.*=+", l.strip()))
        parts.append(text)
        total += len(text.split())
        print(f"   + «{title}»: {len(text.split())} слів")
        if total >= target_words:
            break
    return _check("Вікіпедія", "\n\n".join(parts))


FETCHERS = {"kotsiubynskyi": fetch_kotsiubynskyi,
            "konstytutsiia": fetch_konstytutsiia,
            "wikipedia_ml": fetch_wikipedia}


def ensure_texts(folder="texts"):
    """Завантажує відсутні тексти. Наявні файли (наприклад, ваші власні) не чіпає."""
    os.makedirs(folder, exist_ok=True)
    missing = []
    for name, fetch in FETCHERS.items():
        path = os.path.join(folder, f"{name}.txt")
        if os.path.exists(path) and os.path.getsize(path) > 0:
            print(f"[є]  {path}")
            continue
        print(f"[...] завантажую {name} ...")
        try:
            text = fetch()
            with open(path, "w", encoding="utf-8") as f:
                f.write(text)
            print(f"[ok] {path}: {len(text.split())} слів")
        except Exception as err:
            print(f"[!!] {name}: не вдалося завантажити ({err})")
            missing.append(name)
    if missing:
        try:
            from google.colab import files
            print("\nЗавантажте вручну файли:", ", ".join(f"{m}.txt" for m in missing))
            for fname, data in files.upload().items():
                with open(os.path.join(folder, fname), "wb") as f:
                    f.write(data)
        except ImportError:
            print("Покладіть файли", ", ".join(f"{m}.txt" for m in missing), "у папку", folder)


FUNCTION_WORDS = {
    "PREP": "до за на про під перед серед поза понад після проти протягом навколо об",
    "CONJ": "щоб бо хоча",
    "PRCL": "ж же теж тільки",
    "ADVB": "вже ще",
}
OVERRIDES = {w: (w, pos) for pos, ws in FUNCTION_WORDS.items() for w in ws.split()}
OVERRIDES.update({w: ("дані", "NOUN") for w in ("дані", "даних", "даним", "даними")})
OVERRIDES.update({w: ("я", "NPRO") for w in ("мене", "мені", "мною")})
EDITORIAL_ABBR = {"англ", "лат", "нім", "фр", "рос", "пол", "гр"}
CYRILLIC = re.compile(r"[А-ЩЬЮЯЄІЇҐа-щьюяєіїґ]")


def analyse(word):
    """-> (лема, частина мови) з виправленнями типових помилок pymorphy."""
    if word in OVERRIDES:
        return OVERRIDES[word]
    parses = MORPH.parse(word)
    p = parses[0]
    if str(p.tag.POS) == "NOUN":
        for q in parses:
            if str(q.tag.POS) == "NPRO":
                return q.normal_form, "NPRO"
    return p.normal_form, (str(p.tag.POS) if p.tag.POS else "UNKN")


def load_text(conn, title, text):
    cur = conn.cursor()
    cur.execute("INSERT INTO texts(title) VALUES (?)", (title,))
    text_id = cur.lastrowid

    text = re.sub(r"\((?:англ|лат|нім|фр|рос|пол)\.[^)]*\)", " ", text)

    parsed, total = [], 0
    for sent in tok.tokenize_sents(text):
        toks = []
        for token in tok.tokenize_words(sent):
            token = token.rstrip(".")
            if not token or not CYRILLIC.search(token):
                continue
            if token.lower() in EDITORIAL_ABBR:
                continue
            toks.append(token)
        if toks:
            parsed.append((sent, toks))
            total += len(toks)


    k = min(MAX_SUBSAMPLES, total // SUBSAMPLE_SIZE)
    if k == 0:
        raise ValueError(f"{title}: менше {SUBSAMPLE_SIZE} слововживань")
    limit = k * SUBSAMPLE_SIZE
    if limit < TARGET:
        print(f"   ! {title}: лише {total} слововживань -> {k} підвибірок "
              f"(методичка: {TARGET}). Додайте тексти.")

    n = 0
    for s_pos, (sent, toks) in enumerate(parsed, 1):
        if n >= limit:
            break
        toks = toks[:limit - n]
        cur.execute("INSERT INTO sentences(text_id, position, content) VALUES (?,?,?)",
                    (text_id, s_pos, sent))
        sentence_id = cur.lastrowid
        rows = []
        for w_pos, token in enumerate(toks, 1):
            n += 1
            clean = token.lower()
            lemma, pos = analyse(clean)
            sub = (n - 1) // SUBSAMPLE_SIZE + 1
            rows.append((sentence_id, text_id, w_pos, token, clean, lemma, pos, sub))
        cur.executemany(
            "INSERT INTO words(sentence_id, text_id, position, word, clean_word, lemma, pos, subsample)"
            " VALUES (?,?,?,?,?,?,?,?)", rows)
    conn.commit()
    return text_id


def build_freq_tables(conn):
    """freq_forms, freq_lemmas, freq_pos: id, абс. частота, частота у підвибірках
    (NULL - такої підвибірки у тексту немає)."""
    parts = conn.execute("SELECT COALESCE(MAX(subsample),1) FROM words").fetchone()[0]
    cols = ",\n    ".join(
        f"CASE WHEN {k} > MAX(m.mx) THEN NULL "
        f"ELSE SUM(CASE WHEN w.subsample = {k} THEN 1 ELSE 0 END) END AS part_{k}"
        for k in range(1, parts + 1))

    specs = {
        "freq_forms":  ("w.text_id, w.clean_word AS wordform, w.lemma, w.pos",
                        "w.text_id, w.clean_word, w.lemma, w.pos"),
        "freq_lemmas": ("w.text_id, w.lemma", "w.text_id, w.lemma"),
        "freq_pos":    ("w.text_id, w.pos",   "w.text_id, w.pos"),
    }
    for name, (select_cols, group_cols) in specs.items():
        conn.execute(f"DROP TABLE IF EXISTS {name}")
        conn.execute(f"""
            CREATE TABLE {name} AS
            SELECT ROW_NUMBER() OVER (ORDER BY w.text_id, COUNT(*) DESC) AS id,
                   {select_cols},
                   COUNT(*) AS abs_freq,
                   {cols}
            FROM words w
            JOIN (SELECT text_id, MAX(subsample) AS mx FROM words GROUP BY text_id) m
                 ON m.text_id = w.text_id
            GROUP BY {group_cols}
            ORDER BY w.text_id, abs_freq DESC
        """)
    conn.commit()
    return parts


Q_POS = """
SELECT t.title, w.pos, COUNT(*) AS cnt
FROM words w JOIN texts t ON t.text_id = w.text_id
GROUP BY t.title, w.pos
ORDER BY t.title, cnt DESC;
"""

Q_POS_SHARE = """
SELECT t.title, w.pos, COUNT(*) AS cnt,
       ROUND(100.0 * COUNT(*) /
             (SELECT COUNT(*) FROM words x WHERE x.text_id = w.text_id), 1) AS pct
FROM words w JOIN texts t ON t.text_id = w.text_id
GROUP BY w.text_id, w.pos
ORDER BY t.title, cnt DESC;
"""


def q_rel_error(col):
    """Відносна похибка частки (95%): delta = 1.96 * sqrt((1-p)/(n*p)) * 100%."""
    return f"""
SELECT t.title, w.{col} AS unit, COUNT(*) AS cnt, tot.n AS sample_size,
       ROUND(100.0 * COUNT(*) / tot.n, 2) AS p_percent,
       ROUND(100.0 * 1.96 * sqrt((1.0 - 1.0 * COUNT(*) / tot.n) / (1.0 * COUNT(*))), 2) AS rel_error_percent
FROM words w
JOIN texts t ON t.text_id = w.text_id
JOIN (SELECT text_id, COUNT(*) AS n FROM words GROUP BY text_id) tot ON tot.text_id = w.text_id
GROUP BY w.text_id, w.{col}
ORDER BY t.title, cnt DESC;
"""


Q_REL_ERROR = q_rel_error("pos")
Q_REL_ERROR_LEMMA = q_rel_error("lemma")
Q_REL_ERROR_FORM = q_rel_error("clean_word")


Q_TFIDF = """
WITH tf AS (SELECT text_id, lemma, COUNT(*) AS n FROM words GROUP BY text_id, lemma),
len AS (SELECT text_id, COUNT(*) AS total FROM words GROUP BY text_id),
df  AS (SELECT lemma, COUNT(DISTINCT text_id) AS df FROM words GROUP BY lemma),
n_docs AS (SELECT COUNT(*) AS n FROM texts)
SELECT t.title, tf.lemma, tf.n,
       ROUND(1.0 * tf.n / len.total, 4) AS tf,
       df.df,
       ROUND(ln(1.0 * n_docs.n / df.df), 4) AS idf,
       ROUND(1.0 * tf.n / len.total * ln(1.0 * n_docs.n / df.df), 4) AS tfidf
FROM tf
JOIN len ON len.text_id = tf.text_id
JOIN df  ON df.lemma = tf.lemma
JOIN texts t ON t.text_id = tf.text_id
CROSS JOIN n_docs
ORDER BY t.title, tfidf DESC, tf.lemma;
"""


Q_TFIDF_SUB = """
WITH d AS (SELECT text_id * 100 + subsample AS doc, text_id, lemma, COUNT(*) AS n
           FROM words GROUP BY doc, lemma),
len AS (SELECT text_id, text_id * 100 + subsample AS doc, COUNT(*) AS total
        FROM words GROUP BY doc),
df AS (SELECT lemma, COUNT(DISTINCT doc) AS df FROM d GROUP BY lemma),
nd AS (SELECT COUNT(*) AS n FROM len),
parts AS (SELECT text_id, COUNT(*) AS k FROM len GROUP BY text_id)
SELECT t.title, d.lemma, SUM(d.n) AS n, df.df,
       ROUND(ln(1.0 * nd.n / df.df), 4) AS idf,
       ROUND(SUM(1.0 * d.n / len.total) / parts.k * ln(1.0 * nd.n / df.df), 5) AS mean_tfidf
FROM d
JOIN len ON len.doc = d.doc
JOIN df ON df.lemma = d.lemma
JOIN parts ON parts.text_id = d.text_id
JOIN texts t ON t.text_id = d.text_id
CROSS JOIN nd
GROUP BY d.text_id, d.lemma
ORDER BY t.title, mean_tfidf DESC, d.lemma;
"""

Q_TOP_LEMMAS = """
SELECT t.title, f.lemma, f.abs_freq
FROM freq_lemmas f JOIN texts t ON t.text_id = f.text_id
ORDER BY t.title, f.abs_freq DESC, f.lemma;
"""


Q_TTR = """
SELECT t.title,
       COUNT(*)                     AS tokens,
       COUNT(DISTINCT w.clean_word) AS word_forms,
       COUNT(DISTINCT w.lemma)      AS lemmas,
       ROUND(1.0 * COUNT(DISTINCT w.lemma) / COUNT(*), 3) AS ttr_lemma,
       ROUND(AVG(LENGTH(w.clean_word)), 2)                AS avg_word_len,
       MAX(w.subsample)             AS subsamples,
       ROUND(COUNT(DISTINCT w.lemma) / sqrt(1.0 * COUNT(*)), 2) AS guiraud_lemma
FROM words w JOIN texts t ON t.text_id = w.text_id
GROUP BY w.text_id;
"""

Q_SENT_LEN = """
SELECT t.title, COUNT(*) AS sentences,
       ROUND(AVG(c), 2) AS avg_words_per_sentence, MAX(c) AS max_words, MIN(c) AS min_words
FROM (SELECT text_id, sentence_id, COUNT(*) AS c FROM words GROUP BY sentence_id) s
JOIN texts t ON t.text_id = s.text_id
GROUP BY s.text_id;
"""

Q_HAPAX = """
SELECT t.title, COUNT(*) AS hapax_lemmas
FROM (SELECT text_id, lemma FROM words GROUP BY text_id, lemma HAVING COUNT(*) = 1) h
JOIN texts t ON t.text_id = h.text_id
GROUP BY h.text_id;
"""

Q_WORDLEN = """
SELECT LENGTH(clean_word) AS len, COUNT(*) AS cnt FROM words GROUP BY len ORDER BY len;
"""


def show(conn, title, sql, limit_per_group=None):
    print(f"\n=== {title} ===")
    cur = conn.execute(sql)
    print(" | ".join(d[0] for d in cur.description))
    print("-" * 60)
    seen = {}
    for row in cur:
        if limit_per_group:
            seen[row[0]] = seen.get(row[0], 0) + 1
            if seen[row[0]] > limit_per_group:
                continue
        print(" | ".join(str(x) for x in row))


def export_csv(conn, name, sql):
    os.makedirs(OUT_DIR, exist_ok=True)
    cur = conn.execute(sql)
    with open(os.path.join(OUT_DIR, f"{name}.csv"), "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow([d[0] for d in cur.description])
        w.writerows(cur)


def _md(headers, rows):
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    out += ["| " + " | ".join(str(x) for x in r) + " |" for r in rows]
    return "\n".join(out)


def _by_title(conn, sql, limit=None):
    res = {}
    for row in conn.execute(sql):
        res.setdefault(row[0], [])
        if limit is None or len(res[row[0]]) < limit:
            res[row[0]].append(row[1:])
    return res


def write_report(conn, parts):
    names = [r[0] for r in conn.execute("SELECT title FROM texts ORDER BY text_id")]
    summ = {r[0]: r[1:] for r in conn.execute(Q_TTR)}
    sents = {r[0]: r[1:] for r in conn.execute(Q_SENT_LEN)}
    hapax = {r[0]: r[1] for r in conn.execute(Q_HAPAX)}
    relerr = _by_title(conn, Q_REL_ERROR)
    relerr_l = _by_title(conn, Q_REL_ERROR_LEMMA, 10)
    share = _by_title(conn, Q_POS_SHARE)
    tfidf = _by_title(conn, Q_TFIDF, 10)
    tfidf_s = _by_title(conn, Q_TFIDF_SUB, 10)
    lemmas = _by_title(conn, Q_TOP_LEMMAS, 10)
    L = []
    L.append("# Автоматичне укладання ЧС: звіт (чернетка з автоматично підставленими даними)\n")
    L.append("## І. Вступна частина\n")
    L.append("### 1.1. Лінгвістична однорідність вибірок\n")
    rows = []
    for i, n in enumerate(names, 1):
        m = META.get(n, (n, "[тема]", "[жанр]", "[стиль]", "[джерело]"))
        rows.append((i, m[0], m[1], m[2], m[3], m[4]))
    L.append(_md(["№", "Вибірка", "Тема", "Жанр", "Стиль", "Джерело"], rows))
    L.append("\nКожна вибірка складена з текстів однієї мови, одного стилю й жанру, "
             "тому вважається лінгвістично однорідною.\n")
    L.append("### 1.2. Статистична однорідність вибірок\n")
    L.append("Вибірки суцільні: взято повний текст (або послідовні розділи / статті) без пропусків; "
             f"кожну обрізано до кратного {SUBSAMPLE_SIZE} обсягу (не більше {TARGET}) і поділено на "
             f"рівні підвибірки по {SUBSAMPLE_SIZE} слововживань.\n")
    rows = []
    for n in names:
        tokens, forms, lem, ttr, avgl, subs, gir = summ[n]
        ok = [r for r in relerr[n] if r[3] >= 5.0]
        worst = max((r[4] for r in ok), default=float("nan"))
        rows.append((n, tokens, forms, lem, subs, worst))
    L.append(_md(["Вибірка", "Слововживань", "Словоформ", "Лем", "Підвибірок",
                  "Макс. відн. похибка для ЧМ з часткою ≥ 5 %, %"], rows))
    L.append("\nВідносна похибка частки: δ = 1,96·√((1−p)/(n·p)) (рівень довіри 95 %). "
             "Для рідкісних одиниць похибка велика, тому висновки про них обережніші.\n")
    L.append("Відносна похибка за частинами мови:\n")
    for n in names:
        L.append(f"**{n}**\n")
        L.append(_md(["Частина мови", "Слововживань", "Частка, %", "δ, %"],
                     [(f"{POS_UK.get(r[0], r[0])} ({r[0]})", r[1], r[3], r[4]) for r in relerr[n]]))
        L.append("")
    L.append("Відносна похибка для 10 найчастотніших лем:\n")
    for n in names:
        L.append(f"**{n}**\n")
        L.append(_md(["Лема", "Слововживань", "Частка, %", "δ, %"],
                     [(r[0], r[1], r[3], r[4]) for r in relerr_l[n]]))
        L.append("")
    L.append("## ІІ. Основна частина\n")
    L.append("### 2.1. Методика\n")
    L.append("Токенізація - `tokenize_uk`; лематизація й частини мови - `pymorphy2/pymorphy3` "
             "(перший розбір слова); база даних - `sqlite3`. Пунктуація, числа й латиниця відкинуті, "
             "повтори слів зберігаються. Для типових помилок pymorphy (службові слова «до», «про», "
             "«на», «за» тощо; «дані» -> «даний»; «мене/мені» -> «мен») задано ручні виправлення. "
             "Неповний хвіст тексту (<1000 слів) відкинуто, тож усі підвибірки рівні.\n")
    L.append("### 2.2. Структура БД (ER-модель)\n")
    L.append("`texts (text_id, title)` 1—* `sentences (sentence_id, text_id, position, content)` "
             "1—* `words (word_id, sentence_id, text_id, position, word, clean_word, lemma, pos, subsample)`.\n")
    L.append("Частотні словники (запит `GROUP BY`): `freq_forms`, `freq_lemmas`, `freq_pos`; "
             f"у кожному - id, абсолютна частота та частота в підвибірках `part_1 ... part_{parts}`.\n")
    cnt = {r[0]: r[1:] for r in conn.execute(
        "SELECT t.title, (SELECT COUNT(*) FROM freq_forms f WHERE f.text_id=t.text_id),"
        "(SELECT COUNT(*) FROM freq_lemmas f WHERE f.text_id=t.text_id),"
        "(SELECT COUNT(*) FROM freq_pos f WHERE f.text_id=t.text_id) FROM texts t")}
    L.append(_md(["Вибірка", "Записів у ЧС словоформ", "у ЧС лем", "у ЧС частин мови"],
                 [(n, *cnt[n]) for n in names]))
    L.append("\n### 2.3. Частини мови\n")
    pos_all = sorted({r[0] for n in names for r in share[n]},
                     key=lambda p: -sum(r[1] for n in names for r in share[n] if r[0] == p))
    table = []
    for p in pos_all:
        row = [f"{POS_UK.get(p, p)} ({p})"]
        for n in names:
            hit = next((r for r in share[n] if r[0] == p), None)
            row.append(f"{hit[1]} ({hit[2]} %)" if hit else "0")
        table.append(row)
    L.append(_md(["Частина мови"] + names, table))
    L.append("\n### 2.4. Найчастотніші леми\n")
    for n in names:
        L.append(f"**{n}**: " + ", ".join(f"{r[0]} ({r[1]})" for r in lemmas[n]) + "\n")
    L.append("### 2.5. TF-IDF (документ = текст; топ-10 лем)\n")
    L.append("tf = n/N, idf = ln(D/df), D - кількість текстів.\n")
    for n in names:
        L.append(f"**{n}**\n")
        L.append(_md(["Лема", "n", "TF", "df", "IDF", "TF-IDF"], tfidf[n]))
        L.append("")
    L.append("### 2.6. TF-IDF (документ = підвибірка; топ-10 лем за середнім TF-IDF)\n")
    for n in names:
        L.append(f"**{n}**\n")
        L.append(_md(["Лема", "n", "df", "IDF", "сер. TF-IDF"], tfidf_s[n]))
        L.append("")
    L.append("### 2.7. Додаткові показники\n")
    rows = []
    for n in names:
        tokens, forms, lem, ttr, avgl, subs, gir = summ[n]
        s = sents[n]
        rows.append((n, ttr, gir, avgl, s[0], s[1], hapax.get(n, 0)))
    L.append(_md(["Вибірка", "TTR (лем/слововж.)", "Індекс Гіро (лем)", "Сер. довжина слова",
                  "Речень", "Сер. слів у реченні", "Hapax (лем)"], rows))
    L.append("\n## Висновки (чернетка)\n")
    for n in names:
        top = share[n][0]
        L.append(f"- У вибірці «{n}» найчастіша частина мови - {POS_UK.get(top[0], top[0])} "
                 f"({top[2]} %), індекс Гіро - {summ[n][6]}, середня довжина речення - {sents[n][1]} слів.")
    L.append("- [Допишіть власні спостереження про стильові відмінності.]")
    L.append("- Недоліки: pymorphy обирає перший розбір слова, тому можливі помилки лематизації "
             "омонімів; частина слів отримує позначку UNKN.")
    os.makedirs(OUT_DIR, exist_ok=True)
    with open(os.path.join(OUT_DIR, "report.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(L))


def cli_txt_files():
    return [a for a in sys.argv[1:] if a.lower().endswith(".txt") and os.path.isfile(a)]


def find_paths():
    return cli_txt_files() or sorted(glob.glob("texts/*.txt"))


def main():
    if not cli_txt_files():
        ensure_texts()
    texts = {}
    for p in find_paths():
        key = os.path.splitext(os.path.basename(p))[0].split("__")[0]
        with open(p, encoding="utf-8") as f:
            texts[key] = texts.get(key, "") + "\n" + f.read()
    if not texts:
        print("Тексти не знайдено - використовую вбудовані приклади.")
        texts = SAMPLE_TEXTS

    if os.path.exists("texts.db"):
        os.remove("texts.db")
    conn = sqlite3.connect("texts.db")
    conn.create_function("ln", 1, math.log)
    conn.create_function("sqrt", 1, math.sqrt)
    conn.executescript(SCHEMA)

    for title, text in texts.items():
        load_text(conn, title, text)
    parts = build_freq_tables(conn)

    show(conn, "1. Обсяг, різноманітність (TTR, Гіро), довжина слова, підвибірки", Q_TTR)
    show(conn, "2. Кількість слововживань за частинами мови", Q_POS)
    show(conn, "3. Частки частин мови, %", Q_POS_SHARE)
    show(conn, "4a. Відносна похибка частки частини мови (95%)", Q_REL_ERROR)
    show(conn, "4b. Відносна похибка частки лем (топ-10)", Q_REL_ERROR_LEMMA, limit_per_group=10)
    show(conn, "4c. Відносна похибка частки словоформ (топ-10)", Q_REL_ERROR_FORM, limit_per_group=10)
    show(conn, "5a. TF-IDF лем, документ = текст (топ-10)", Q_TFIDF, limit_per_group=10)
    show(conn, "5b. TF-IDF лем, документ = підвибірка (топ-10)", Q_TFIDF_SUB, limit_per_group=10)
    show(conn, "6. ЧС лем (топ-10 для кожного тексту)", Q_TOP_LEMMAS, limit_per_group=10)
    show(conn, "7. Довжина речень (у словах)", Q_SENT_LEN)
    show(conn, "8. Hapax legomena (леми, що трапились 1 раз)", Q_HAPAX)
    show(conn, "9. Розподіл за довжиною слова (усі тексти)", Q_WORDLEN)

    exports = {
        "01_summary": Q_TTR, "02_pos_counts": Q_POS, "03_pos_share": Q_POS_SHARE,
        "04a_rel_error_pos": Q_REL_ERROR, "04b_rel_error_lemma": Q_REL_ERROR_LEMMA,
        "04c_rel_error_form": Q_REL_ERROR_FORM,
        "05a_tfidf_text": Q_TFIDF, "05b_tfidf_subsample": Q_TFIDF_SUB,
        "06_sentence_length": Q_SENT_LEN, "07_hapax": Q_HAPAX, "08_word_length": Q_WORDLEN,
        "10_freq_forms": "SELECT t.title, f.* FROM freq_forms f JOIN texts t USING(text_id)",
        "11_freq_lemmas": "SELECT t.title, f.* FROM freq_lemmas f JOIN texts t USING(text_id)",
        "12_freq_pos": "SELECT t.title, f.* FROM freq_pos f JOIN texts t USING(text_id)",
    }
    for name, sql in exports.items():
        export_csv(conn, name, sql)
    write_report(conn, parts)
    conn.close()
    print(f"\nЧернетка звіту: {OUT_DIR}/report.md")
    print(f"Підвибірок на текст: до {parts}. БД: texts.db, таблиці CSV: {OUT_DIR}/")


if __name__ == "__main__":
    main()

[є]  texts/kotsiubynskyi.txt
[є]  texts/konstytutsiia.txt
[є]  texts/wikipedia_ml.txt
   ! konstytutsiia: лише 3997 слововживань -> 3 підвибірок (методичка: 20000). Додайте тексти.
   ! kotsiubynskyi: лише 3900 слововживань -> 3 підвибірок (методичка: 20000). Додайте тексти.
   ! wikipedia_ml: лише 6625 слововживань -> 6 підвибірок (методичка: 20000). Додайте тексти.

=== 1. Обсяг, різноманітність (TTR, Гіро), довжина слова, підвибірки ===
title | tokens | word_forms | lemmas | ttr_lemma | avg_word_len | subsamples | guiraud_lemma
------------------------------------------------------------
konstytutsiia | 3000 | 1143 | 763 | 0.254 | 6.55 | 3 | 13.93
kotsiubynskyi | 3000 | 1490 | 1127 | 0.376 | 4.55 | 3 | 20.58
wikipedia_ml | 6000 | 2517 | 1649 | 0.275 | 6.74 | 6 | 21.29

=== 2. Кількість слововживань за частинами мови ===
title | pos | cnt
------------------------------------------------------------
konstytutsiia | NOUN | 1377
konstytutsiia | ADJF | 463
konstytutsiia | PREP | 278
kons